In [0]:
from pyspark.sql.functions import col
from pyspark.sql.functions import current_timestamp, input_file_name, col, regexp_extract
from pyspark.sql.types import *


schema = StructType([

    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("email", StringType(), True),
    StructField("phone", StringType(), True),
    StructField("country", StringType(), True),

    # mixed date format → keep as string in Bronze
    StructField("signup_date", StringType(), True),
    StructField("last_login", StringType(), True),

    StructField("salary", DoubleType(), True),
    StructField("is_active", BooleanType(), True),

    # array of strings
    StructField("skills", ArrayType(StringType()), True),

    # nested struct: address
    StructField("address", StructType([
        StructField("city", StringType(), True),
        StructField("pincode", IntegerType(), True),

        StructField("coordinates", StructType([
            StructField("lat", DoubleType(), True),
            StructField("lng", DoubleType(), True)
        ]), True)
    ]), True),

    # dynamic keys → use MapType
    StructField("projects", MapType(StringType(), StringType()), True),

    # array of struct
    StructField("transactions", ArrayType(
        StructType([
            StructField("txn_id", StringType(), True),
            StructField("amount", DoubleType(), True),
            StructField("date", StringType(), True)
        ])
    ), True)

])

In [0]:

df = (
    spark.read.format("json")
    .schema(schema)
    .option("columnNameOfCorruptRecord", "_corrupt_record")
    .option("multiline", "true")
    .load('/Volumes/temp/0_dump/temp/json_data/')
)

df = df.withColumns({
    "ingestion_time": current_timestamp(),
    "filename" : regexp_extract(col("_metadata.file_path"), r'([^/]+$)', 1)
})

display(df.head(5))


In [0]:
# df.write.mode('overwrite').saveAsTable('temp.1_bronze.user_dataset')
df.write.format("delta").mode("append").saveAsTable("temp.1_bronze.user_dataset")

In [0]:
# df_read = spark.sql("SELECT * FROM temp.1_bronze.user_dataset")
df_read = spark.read.table("temp.1_bronze.user_dataset")
display(df_read.select(col("id")).distinct())


In [0]:
%sql
-- drop table if exists temp.1_bronze.user_dataset;
-- drop table if exists temp.2_silver.user_dataset;
-- drop table if exists temp.3_gold.user_dataset;
